# pymatchit — Example Workflow

This notebook is a self-contained walkthrough of **pymatchit** — a Python port of R's `MatchIt` package. It covers:

1. **Dataset** — the built-in Lalonde (1986) dataset
2. **Matching methods** — `nearest`, `optimal`, `full`, `exact`, `subclass`, `cem`, `genetic`, `cardinality`
3. **Distance / propensity-score estimators** — `glm`, `cbps`, `mahalanobis`, `randomforest`, `gbm`
4. **All plot types** — `balance` (Love plot), `propensity`, `ecdf`, `qq`, `jitter`
5. **Diagnostics** — `summary()`, `matches()`, `compute_ks_statistics`, `compute_effective_sample_size`

---

> **Quick start** — run every cell top-to-bottom inside your activated virtual environment:
> ```bash
> .\pymatchit_venv\Scripts\activate
> jupyter notebook example_workflow.ipynb
> ```

## 0  Imports & Matplotlib inline display

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Core pymatchit API
from pymatchit import MatchIt, load_lalonde
from pymatchit import compute_ks_statistics, compute_effective_sample_size

# Reproducibility seed used throughout
SEED = 42

---
## 1  Dataset

We use the built-in **Lalonde (1986)** dataset — the canonical benchmark for causal-inference methods.

| Column | Description |
|--------|-------------|
| `treat` | Treatment indicator (1 = job-training program) |
| `age` | Age in years |
| `educ` | Years of education |
| `black` | Race = Black (binary) |
| `hispan` | Hispanic (binary) |
| `married` | Married (binary) |
| `nodegree` | No high-school degree (binary) |
| `re74` | Real earnings in 1974 (USD) |
| `re75` | Real earnings in 1975 (USD) |
| `re78` | Real earnings in 1978 — **outcome** (not used for matching) |

In [ ]:
df = load_lalonde()
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Quick peek at treatment balance in the raw data
df.groupby('treat')[['age', 'educ', 're74', 're75']].mean().round(2)

---
## 2  Convenience helper

We define a small helper that fits a `MatchIt` model, prints the summary, and returns the fitted model — so each section stays concise.

In [ ]:
FORMULA = "treat ~ age + educ + black + hispan + married + nodegree + re74 + re75"

PRETTY = {
    'age':      'Age',
    'educ':     'Education (yrs)',
    'black':    'Black',
    'hispan':   'Hispanic',
    'married':  'Married',
    'nodegree': 'No Degree',
    're74':     'Earnings 1974',
    're75':     'Earnings 1975',
}

def fit_and_summarize(m: MatchIt) -> MatchIt:
    """Fit the model and display summary tables."""
    m.fit(FORMULA)
    stats = m.summary(print_output=False)
    print("--- Sample Sizes ---")
    display(stats['sample_sizes'])
    print("\n--- Balance (Matched) ---")
    display(stats['matched'][['Means Treated', 'Means Control', 'Std. Mean Diff.', 'Var Ratio']].round(3))
    return m

---
## 3  All Five Plot Types

We first fit a single baseline model so we can showcase every plot type in one place.

In [ ]:
# Baseline: 1:1 nearest-neighbour matching with logistic propensity score
m_base = MatchIt(df, method='nearest', distance='glm', ratio=1, random_state=SEED)
m_base = fit_and_summarize(m_base)

### 3.1  Balance / Love Plot

The Love plot shows the **Absolute Standardized Mean Difference (ASMD)** for each covariate before and after matching.  
All points should be left of the dashed threshold (default: 0.1) after matching.

In [ ]:
ax = m_base.plot(type='balance', var_names=PRETTY)
plt.show()

### 3.2  Propensity Score Distribution

Shows the **kernel density estimate** of propensity scores for treated vs. control units — before and after matching.  
Good overlap is essential for valid causal inference.

In [ ]:
ax = m_base.plot(type='propensity')
plt.show()

### 3.3  eCDF Plot

Plots the **Empirical Cumulative Distribution Function** for a single covariate.  
Dashed lines = raw data; solid lines = matched data.  
Overlapping lines indicate good covariate balance.

In [ ]:
ax = m_base.plot(type='ecdf', variable='age')
plt.show()

In [ ]:
# Another covariate
ax = m_base.plot(type='ecdf', variable='re74')
plt.show()

### 3.4  QQ Plot

Compares the **quantile distributions** of treated vs. control groups covariate by covariate.  
Points along the 45° diagonal indicate equal distributions.

In [ ]:
axes = m_base.plot(type='qq', variable='age')
plt.show()

In [ ]:
axes = m_base.plot(type='qq', variable='re74')
plt.show()

### 3.5  Jitter Plot

Shows which units were **matched vs. unmatched**, plotted by propensity score.  
Solid dots = matched; hollow dots = unmatched.

In [ ]:
ax = m_base.plot(type='jitter')
plt.show()

---
## 4  Matching Methods

pymatchit supports 8 matching algorithms. We try each on the Lalonde data.

### 4.1  Nearest Neighbour (1:1)

Greedy 1:1 nearest-neighbour matching on the propensity score.
- `m_order='largest'` — treated units with the highest propensity score are matched first (default).
- `replace=False` — no replacement (each control unit used at most once).

In [ ]:
m_nn = MatchIt(df, method='nearest', distance='glm', ratio=1,
               replace=False, m_order='largest', random_state=SEED)
m_nn = fit_and_summarize(m_nn)
m_nn.plot(type='balance', var_names=PRETTY)
plt.title('Nearest Neighbour — Balance'); plt.show()

### 4.2  Nearest Neighbour with Caliper

Adds a **caliper** (±0.1 standard deviations of the distance measure).  
Treated units with no suitable control within the caliper are **not** matched.

In [ ]:
m_caliper = MatchIt(df, method='nearest', distance='glm', ratio=1,
                    caliper=0.1, replace=False, random_state=SEED)
m_caliper = fit_and_summarize(m_caliper)
m_caliper.plot(type='balance', var_names=PRETTY)
plt.title('Nearest Neighbour with Caliper 0.1 — Balance'); plt.show()

### 4.3  Nearest Neighbour with Replacement (k=2)

`replace=True` allows control units to be reused; `ratio=2` finds two controls per treated unit.

In [ ]:
m_k2 = MatchIt(df, method='nearest', distance='glm', ratio=2,
               replace=True, random_state=SEED)
m_k2 = fit_and_summarize(m_k2)
m_k2.plot(type='balance', var_names=PRETTY)
plt.title('Nearest Neighbour k=2 with Replacement — Balance'); plt.show()

### 4.4  Optimal Matching

Minimises the **total** matched distance globally (not greedy).  
Produces the same number of pairs as nearest-neighbour but with a lower total distance.

In [ ]:
m_opt = MatchIt(df, method='optimal', distance='glm', ratio=1, random_state=SEED)
m_opt = fit_and_summarize(m_opt)
m_opt.plot(type='balance', var_names=PRETTY)
plt.title('Optimal Matching — Balance'); plt.show()

### 4.5  Full Matching

Creates one or more subclasses, each containing at least one treated AND one control unit.  
**All** observations are retained, with fractional weights.

In [ ]:
m_full = MatchIt(df, method='full', distance='glm', random_state=SEED)
m_full = fit_and_summarize(m_full)
m_full.plot(type='balance', var_names=PRETTY)
plt.title('Full Matching — Balance'); plt.show()

### 4.6  Subclassification

Divides the propensity score range into `subclass=6` bins.  
All units within a subclass are retained (weighted proportionally to group size).

In [ ]:
m_sub = MatchIt(df, method='subclass', distance='glm', subclass=6, random_state=SEED)
m_sub = fit_and_summarize(m_sub)
m_sub.plot(type='balance', var_names=PRETTY)
plt.title('Subclassification (6 bins) — Balance'); plt.show()

### 4.7  Exact Matching

Matches units that are **identical** on the specified variable(s).  
Works best for discrete/categorical covariates.

In [ ]:
# Exact match on binary race variables
m_exact = MatchIt(df, method='exact', random_state=SEED)
m_exact = fit_and_summarize(m_exact)
m_exact.plot(type='balance', var_names=PRETTY)
plt.title('Exact Matching — Balance'); plt.show()

### 4.8  Coarsened Exact Matching (CEM)

Coarsens continuous variables into bins, then exact-matches on the coarsened values.  
Pass `cutpoints` to control how bins are formed (default: Sturges' rule).

In [ ]:
m_cem = MatchIt(df, method='cem', random_state=SEED)
m_cem = fit_and_summarize(m_cem)
m_cem.plot(type='balance', var_names=PRETTY)
plt.title('CEM — Balance'); plt.show()

### 4.9  Genetic Matching

Uses a genetic algorithm to **optimise the weighting** of covariates in the Mahalanobis distance,  
maximising balance automatically. Computationally heavier; reduce `pop_size`/`max_generations` for speed.

In [ ]:
m_gen = MatchIt(df, method='genetic', ratio=1, pop_size=30, max_generations=20,
                random_state=SEED)
m_gen = fit_and_summarize(m_gen)
m_gen.plot(type='balance', var_names=PRETTY)
plt.title('Genetic Matching — Balance'); plt.show()

### 4.10  Cardinality Matching

Finds the **largest matched sample** such that each covariate's SMD is within a given tolerance (`std_tols`).  
Solves an optimisation problem — may return fewer matches than other methods.

In [ ]:
m_card = MatchIt(df, method='cardinality', std_tols=0.1, random_state=SEED)
m_card = fit_and_summarize(m_card)
m_card.plot(type='balance', var_names=PRETTY)
plt.title('Cardinality Matching — Balance'); plt.show()

---
## 5  Distance Estimators

All `method='nearest'` below; only the `distance=` argument changes.

### 5.1  GLM (Logistic Regression) — default

In [ ]:
m_glm = MatchIt(df, method='nearest', distance='glm', ratio=1, random_state=SEED)
m_glm.fit(FORMULA)
m_glm.plot(type='propensity')
plt.suptitle('GLM Propensity Score', y=1.02, fontweight='bold')
plt.show()

### 5.2  CBPS (Covariate Balancing Propensity Score)

Jointly optimises prediction accuracy **and** covariate balance.

In [ ]:
m_cbps = MatchIt(df, method='nearest', distance='cbps', ratio=1, random_state=SEED)
m_cbps.fit(FORMULA)
m_cbps.plot(type='propensity')
plt.suptitle('CBPS Propensity Score', y=1.02, fontweight='bold')
plt.show()

### 5.3  Mahalanobis Distance

Matches directly on the multivariate Mahalanobis distance — no propensity score needed.

> ⚠️ `propensity` and `jitter` plots are unavailable for Mahalanobis matching.

In [ ]:
m_mah = MatchIt(df, method='nearest', distance='mahalanobis', ratio=1, random_state=SEED)
m_mah = fit_and_summarize(m_mah)
m_mah.plot(type='balance', var_names=PRETTY)
plt.title('Mahalanobis Distance — Balance'); plt.show()

### 5.4  Random Forest

In [ ]:
m_rf = MatchIt(df, method='nearest', distance='randomforest', ratio=1,
               distance_options={'n_estimators': 300, 'max_depth': 5},
               random_state=SEED)
m_rf.fit(FORMULA)
m_rf.plot(type='propensity')
plt.suptitle('Random Forest Propensity Score', y=1.02, fontweight='bold')
plt.show()
m_rf.plot(type='balance', var_names=PRETTY)
plt.title('Random Forest — Balance'); plt.show()

### 5.5  Gradient Boosting (GBM)

In [ ]:
m_gbm = MatchIt(df, method='nearest', distance='gbm', ratio=1,
                distance_options={'n_estimators': 200, 'learning_rate': 0.1},
                random_state=SEED)
m_gbm.fit(FORMULA)
m_gbm.plot(type='propensity')
plt.suptitle('GBM Propensity Score', y=1.02, fontweight='bold')
plt.show()
m_gbm.plot(type='balance', var_names=PRETTY)
plt.title('GBM — Balance'); plt.show()

### 5.6  User-Supplied Propensity Scores

You can pass a pre-computed `numpy` array or `pandas` Series as the `distance` argument.

In [ ]:
from sklearn.linear_model import LogisticRegression
import patsy

# Fit a custom logistic regression outside pymatchit
y_arr, X_arr = patsy.dmatrices(FORMULA, df, return_type='dataframe')
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_arr, y_arr.iloc[:, 0])
custom_ps = lr.predict_proba(X_arr)[:, 1]

m_custom = MatchIt(df, method='nearest', distance=custom_ps, ratio=1, random_state=SEED)
m_custom.fit(FORMULA)
m_custom.plot(type='propensity')
plt.suptitle('User-Supplied Propensity Score', y=1.02, fontweight='bold')
plt.show()

---
## 6  Advanced Options

### 6.1  Discard Units Outside Common Support

`discard='both'` removes treated and control units whose propensity scores are outside the other group's range.

In [ ]:
m_discard = MatchIt(df, method='nearest', distance='glm', ratio=1,
                    discard='both', random_state=SEED)
m_discard = fit_and_summarize(m_discard)
m_discard.plot(type='jitter')
plt.title('Common-Support Discarding — Jitter'); plt.show()

### 6.2  Exact + Propensity Score Matching

The `exact` argument forces exact equality on the listed variables **within** the propensity score matching.

In [ ]:
m_exact_ps = MatchIt(df, method='nearest', distance='glm', ratio=1,
                     exact=['married'], random_state=SEED)
m_exact_ps = fit_and_summarize(m_exact_ps)
m_exact_ps.plot(type='balance', var_names=PRETTY)
plt.title('Nearest + Exact on Married — Balance'); plt.show()

### 6.3  Mahalanobis within PS Caliper (`mahvars`)

`mahvars` uses the propensity score as a caliper and then ranks matches by Mahalanobis distance on the specified variables.

In [ ]:
m_mah_ps = MatchIt(df, method='nearest', distance='glm', ratio=1,
                   mahvars=['age', 'educ', 're74'], caliper=0.2, random_state=SEED)
m_mah_ps = fit_and_summarize(m_mah_ps)
m_mah_ps.plot(type='balance', var_names=PRETTY)
plt.title('Mahalanobis within PS Caliper — Balance'); plt.show()

### 6.4  ATE Estimand

By default, pymatchit targets the **ATT** (Average Treatment effect on the Treated).  
Set `estimand='ATE'` to target the population average treatment effect.

In [ ]:
m_ate = MatchIt(df, method='full', distance='glm', estimand='ATE', random_state=SEED)
m_ate = fit_and_summarize(m_ate)
m_ate.plot(type='balance', var_names=PRETTY)
plt.title('Full Matching — ATE Estimand — Balance'); plt.show()

---
## 7  Diagnostics

### 7.1  `summary()` — Full Balance Table

In [ ]:
stats = m_base.summary(print_output=False)

print("=== Sample Sizes ===")
display(stats['sample_sizes'])

print("\n=== Unmatched Balance ===")
display(stats['unmatched'][['Means Treated', 'Means Control', 'Std. Mean Diff.', 'Var Ratio']].round(3))

print("\n=== Matched Balance ===")
display(stats['matched'][['Means Treated', 'Means Control', 'Std. Mean Diff.', 'Var Ratio']].round(3))

### 7.2  `matches()` — Retrieve Match Pairs

In [ ]:
print("Long format:")
display(m_base.matches(format='long').head(10))

print("\nWide format:")
display(m_base.matches(format='wide').head(10))

### 7.3  KS Statistics

In [ ]:
covariates = ['age', 'educ', 're74', 're75']
ks = compute_ks_statistics(
    data=m_base.data,
    covariates=covariates,
    treatment_col='treat',
    weights=m_base.weights
)
display(ks.round(4))

### 7.4  Effective Sample Size (ESS)

In [ ]:
ess = compute_effective_sample_size(
    weights=m_base.weights,
    treatment=m_base.data['treat']
)
print("Effective Sample Size after matching:")
for group, val in ess.items():
    print(f"  {group}: {val}")

---
## 8  Complete Side-by-Side Method Comparison

We compare the **maximum ASMD after matching** across several key methods to pick the best fit.

In [ ]:
methods_to_compare = {
    'Nearest (GLM)':    MatchIt(df, method='nearest',  distance='glm',          ratio=1, random_state=SEED),
    'Optimal (GLM)':    MatchIt(df, method='optimal',  distance='glm',          ratio=1, random_state=SEED),
    'Full (GLM)':       MatchIt(df, method='full',     distance='glm',                   random_state=SEED),
    'CBPS':             MatchIt(df, method='nearest',  distance='cbps',         ratio=1, random_state=SEED),
    'Random Forest':    MatchIt(df, method='nearest',  distance='randomforest', ratio=1,
                                distance_options={'n_estimators': 200}, random_state=SEED),
    'Mahalanobis':      MatchIt(df, method='nearest',  distance='mahalanobis',  ratio=1, random_state=SEED),
}

comparison = {}
for name, model in methods_to_compare.items():
    model.fit(FORMULA)
    s = model.summary(print_output=False)
    max_smd = s['matched']['Std. Mean Diff.'].abs().max()
    n_matched = s['sample_sizes'].loc['Matched', 'Treated']
    comparison[name] = {'Max |ASMD| (Matched)': round(max_smd, 4), 'N Matched Treated': n_matched}

comp_df = pd.DataFrame(comparison).T
comp_df = comp_df.sort_values('Max |ASMD| (Matched)')
display(comp_df)

In [ ]:
# Bar chart of max ASMD
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if v < 0.1 else '#e67e22' if v < 0.2 else '#e74c3c'
          for v in comp_df['Max |ASMD| (Matched)']]
bars = ax.barh(comp_df.index, comp_df['Max |ASMD| (Matched)'], color=colors, edgecolor='none', height=0.6)
ax.axvline(0.1, color='grey', linestyle='--', linewidth=1.2, label='Threshold 0.1')
ax.set_xlabel('Maximum |ASMD| After Matching')
ax.set_title('Method Comparison — Worst-Case Balance', fontweight='bold')
ax.legend(frameon=False)
import seaborn as sns; sns.despine()
plt.tight_layout(); plt.show()

---
## 9  Using the Matched Dataset for Outcome Analysis

After matching, use `model.matched_data` (with `weights` column) in your downstream regression.

> Always control for covariates in the outcome model even after matching — this is the **doubly-robust** approach.

In [ ]:
import statsmodels.formula.api as smf

# Re-attach the outcome variable to the matched dataset
matched = m_base.matched_data.copy()
matched['re78'] = df.loc[matched.index, 're78']

# Weighted OLS outcome model
ols = smf.wls(
    're78 ~ treat + age + educ + black + hispan + married + nodegree + re74 + re75',
    data=matched,
    weights=matched['weights']
).fit()

print(f"Estimated ATT (re78): ${ols.params['treat']:,.0f}  "
      f"(95% CI: ${ols.conf_int().loc['treat', 0]:,.0f} – ${ols.conf_int().loc['treat', 1]:,.0f})")
print(f"p-value: {ols.pvalues['treat']:.4f}")

---
## Summary

| Topic | Key APIs |
|-------|----------|
| Create model | `MatchIt(df, method=..., distance=..., ...)` |
| Fit | `.fit("treat ~ age + educ + ...")` |
| Balance summary | `.summary()` |
| Love plot | `.plot(type='balance', var_names=...)` |
| Propensity plot | `.plot(type='propensity')` |
| eCDF plot | `.plot(type='ecdf', variable='age')` |
| QQ plot | `.plot(type='qq', variable='age')` |
| Jitter plot | `.plot(type='jitter')` |
| Match pairs | `.matches(format='long'/'wide')` |
| Matched data | `.matched_data` (DataFrame with `weights` column) |
| KS statistics | `compute_ks_statistics(data, covariates, treatment_col, weights)` |
| ESS | `compute_effective_sample_size(weights, treatment)` |